# Outcome-based evaluation summary

One table counting how many of the verifier's measurements each agent passed,
per dataset — the summary the paper's `outcome_summary` table shows, extended
to all eight datasets.

Each cell is **passes / measured** over the three trials of one agent:

| Column | Passes when |
|---|---|
| Checks | the pass/fail check is true (`CHECK_FIELDS`, one per row of the checks table) |
| Statistics | the agent / reference ratio of a scale metric is within `[0.9, 1.1]` |
| Decoder | the agent / reference validation-accuracy ratio is at least `DECODER_MIN` |
| End-to-end | *(per trial)* every metric measured for that trial passed |

A measurement the verifier never recorded is left out of **both** numbers, so
denominators differ between datasets — Allen2P has no reference for some scale
metrics, and a trial that died early contributes nothing rather than a failure.

The numbers come from `metrics.py`'s own extractors (`_check_values`,
`_scale_supervised_row`, `_ordered_decoder_rows`), so this table and the three
per-dataset tables it summarises cannot drift apart.

In [1]:
# Load metrics.py's helpers and data.
#
# `import metrics` does not work today, so its cells are exec'd here with three
# overrides. Each is a one-line workaround for something broken in the file; once
# those are fixed this whole cell becomes `from metrics import *`.
#
#   1. TRIAL_METRICS_JSON asks for trial_metrics_all.json, which is not in the
#      tree. trial_metrics.json is, and holds all four agents x two prompts.
#   2. PROMPT_SHORT is read by _latex_group_header but defined nowhere, so every
#      LaTeX-emitting cell raises NameError.
#   3. The three `# compare ...` cells call render_metric_table(UNSUPERVISED_DS),
#      which now raises on the empty dataset list. Skipped -- no table needs the
#      figures they draw.

from pathlib import Path

import numpy as np

EVAL = Path.cwd()
METRICS_PY = EVAL / "metrics.py"
FIGURES_DIR = EVAL.parents[1] / "figures"


def load_metrics(path=METRICS_PY):
    """Exec metrics.py and hand back its namespace."""
    src = path.read_text()
    src = src.replace("TRIAL_METRICS_JSON = 'trial_metrics_all.json'",
                      "TRIAL_METRICS_JSON = 'trial_metrics.json'")

    cells, current = [], []
    for line in src.splitlines(keepends=True):
        if line.startswith("# %%"):
            cells.append("".join(current))
            current = []
        current.append(line)
    cells.append("".join(current))

    def is_figure_cell(cell):
        # Identified by the cell's own opening comment. Matching the body text
        # ('with arms_subset(') would also drop the helpers cell, whose
        # docstrings use that idiom as an example.
        body = [l for l in cell.splitlines()
                if l.strip() and not l.startswith("# %%")]
        return bool(body) and body[0].startswith("# compare")

    ns = {"__file__": str(path), "__name__": "metrics_loaded",
          "PROMPT_SHORT": {"minimal": "minimal", "full": "maximal"}}
    exec(compile("".join(c for c in cells if not is_figure_cell(c)),
                 str(path), "exec"), ns)
    return ns


M = load_metrics()

# The two arms the published tables show. ARM_COLUMNS now holds six.
ARMS = [("claude-code", "full"), ("codex", "full")]
M["set_arms"](ARMS)

DATASETS = M["SUPERVISED_DS"]
TRIALS = M["TRIALS_PER_ARM"]
print(f"{len(DATASETS)} datasets, arms: {M['ARM_COLUMNS']}")

Figures will be written to /groups/zhang/home/zhangl5/Data-Format/figures
% requires \usepackage{booktabs, multirow, xcolor}
\begin{table}[t]
\centering
\setlength{\tabcolsep}{4pt}
\begin{tabular}{l l rrr rrr rrr rrr rrr rrr c}
\toprule
 & & \multicolumn{3}{c}{Claude Code (minimal)} & \multicolumn{3}{c}{Claude Code (maximal)} & \multicolumn{3}{c}{Codex (minimal)} & \multicolumn{3}{c}{Codex (maximal)} & \multicolumn{3}{c}{Terminus/Opus (maximal)} & \multicolumn{3}{c}{Terminus/GPT (maximal)} & Reference \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11} \cmidrule(lr){12-14} \cmidrule(lr){15-17} \cmidrule(lr){18-20}
Dataset & Variable & T1 & T2 & T3 & T1 & T2 & T3 & T1 & T2 & T3 & T1 & T2 & T3 & T1 & T2 & T3 & T1 & T2 & T3 & (chance) \\
\midrule
\multirow{5}{*}{Allen2P} & trial\_outcome & \textcolor{orange}{0.813} & \textcolor{orange}{0.825} & 0.906 & 0.919 & 0.991 & \textcolor{orange}{0.838} & 1.011 & 0.953 & \textcolor{orange}{0.811} & 0.905 & \textcolor{orange}{0.836} & \text

In [2]:
# Pass counting.

STAT_BAND = (0.9, 1.1)   # a scale metric "agrees with the reference"
# 0.90, not the verifier's own MIN_ACCURACY_FRAC of 0.95: this is what
# reproduces the published table's counts (Allen2P Claude 9/15, Sosa2024 Codex
# 17/18), whose caption nonetheless says 95%. Worth reconciling before the next
# version of the paper -- at 0.95 those two become 7/15 and 15/18.
DECODER_MIN = 0.90
GREEN = 2 / 3            # the rate the paper prints in green; bold here


def agent_cells(values, arm_index):
    """The TRIALS cells belonging to one arm, from an N_CELLS-long row."""
    return list(values[arm_index * TRIALS:(arm_index + 1) * TRIALS])


def pass_counts(ds):
    """{arm index: {'Checks'/'Statistics'/'Decoder': [bool], 'trials': [bool]}}.

    'trials' backs the End-to-end column: one flag per trial, true when every
    metric measured for that trial passed. Unmeasured cells are skipped, so they
    neither pass nor fail.
    """
    rows = {
        "Checks": ([M["_check_values"](ds, fn) for _lbl, fn in M["CHECK_FIELDS"]],
                   lambda v: v >= 0.5),
        "Statistics": ([M["_scale_supervised_row"](ds, key)[0]
                        for _lbl, key in M["SCALE_FIELDS_LATEX"]],
                       lambda v: STAT_BAND[0] <= v <= STAT_BAND[1]),
        "Decoder": ([vals for _var, vals, _r
                     in M["_ordered_decoder_rows"](ds, supervised=True)],
                    lambda v: v >= DECODER_MIN),
    }

    out = {}
    for arm_index in range(len(M["ARM_COLUMNS"])):
        per_trial = [[] for _ in range(TRIALS)]
        counts = {}
        for name, (metric_rows, passed) in rows.items():
            got = []
            for row in metric_rows:
                for trial, v in enumerate(agent_cells(row, arm_index)):
                    if v is None or np.isnan(v):
                        continue
                    got.append(passed(v))
                    per_trial[trial].append(passed(v))
            counts[name] = got
        counts["trials"] = [all(t) for t in per_trial if t]
        out[arm_index] = counts
    return out

In [3]:
# Render as a markdown pipe table.

CATEGORIES = ["Checks", "Statistics", "Decoder"]


def frac(n_passed, n_measured):
    """"passes/measured", bolded at the rate the paper colours green."""
    if not n_measured:
        return "\u2013"
    cell = f"{n_passed}/{n_measured}"
    return f"**{cell}**" if n_passed / n_measured >= GREEN else cell


def summary_table(datasets=None):
    header = ["Dataset", "Agent", *CATEGORIES, "End-to-end"]
    lines = ["| " + " | ".join(header) + " |",
             "|---|---|" + "---:|" * (len(header) - 2)]
    for ds in (datasets or DATASETS):
        counts = pass_counts(ds)
        for arm_index, arm in enumerate(M["ARM_COLUMNS"]):
            agent = M["AGENT_SHORT"][M["ARM_AGENT"][arm]]
            cells = [frac(sum(counts[arm_index][c]), len(counts[arm_index][c]))
                     for c in CATEGORIES]
            e2e = counts[arm_index]["trials"]
            lines.append("| " + " | ".join(
                [M["display_name"](ds) if arm_index == 0 else "", agent,
                 *cells, frac(sum(e2e), len(e2e))]) + " |")
    return "\n".join(lines)


print(summary_table())

| Dataset | Agent | Checks | Statistics | Decoder | End-to-end |
|---|---|---:|---:|---:|---:|
| Allen2P | Claude | **14/15** | 5/15 | 9/15 | 0/3 |
|  | Codex | **12/15** | 6/15 | 4/15 | 0/3 |
| Chen2024 | Claude | **11/15** | 9/15 | **12/12** | 0/3 |
|  | Codex | **12/15** | **12/15** | **9/12** | 0/3 |
| Hasnain2024 | Claude | **12/15** | 8/15 | **18/18** | 0/3 |
|  | Codex | **12/15** | 7/15 | **18/18** | 0/3 |
| Lee2025 | Claude | **15/15** | **14/15** | **3/3** | **2/3** |
|  | Codex | **15/15** | **13/15** | **3/3** | 1/3 |
| Majnik2025 | Claude | **15/15** | 9/15 | 1/3 | 0/3 |
|  | Codex | **13/14** | 9/15 | **2/2** | 0/3 |
| Sosa2024 | Claude | **13/15** | **15/15** | **16/18** | 0/3 |
|  | Codex | **15/15** | **13/15** | **17/18** | 1/3 |
| Zhang2025 (IBL) | Claude | **14/15** | 7/15 | **10/12** | 0/3 |
|  | Codex | **15/15** | **15/15** | **12/12** | **3/3** |
| Zhong2025 | Claude | **11/15** | **11/15** | 3/12 | 0/3 |
|  | Codex | **12/15** | 9/15 | 5/12 | 0/3 |


In [4]:
# Any subset, in any order.
print(summary_table(["chen2024", "hasnain2024", "zhang2025", "zhong2025"]))

| Dataset | Agent | Checks | Statistics | Decoder | End-to-end |
|---|---|---:|---:|---:|---:|
| Chen2024 | Claude | **11/15** | 9/15 | **12/12** | 0/3 |
|  | Codex | **12/15** | **12/15** | **9/12** | 0/3 |
| Hasnain2024 | Claude | **12/15** | 8/15 | **18/18** | 0/3 |
|  | Codex | **12/15** | 7/15 | **18/18** | 0/3 |
| Zhang2025 (IBL) | Claude | **14/15** | 7/15 | **10/12** | 0/3 |
|  | Codex | **15/15** | **15/15** | **12/12** | **3/3** |
| Zhong2025 | Claude | **11/15** | **11/15** | 3/12 | 0/3 |
|  | Codex | **12/15** | 9/15 | 5/12 | 0/3 |


In [5]:
# Write it out. Uncomment to save alongside the LaTeX tables metrics.py emits.
# (FIGURES_DIR / "outcome_summary.md").write_text(summary_table() + "\n")